# HW2. Пример решения

Ноутбук показывает, как устроено решение, которое мы запускаем на скрытом
тесте. Функции лежат рядом, в `solution_utils.py`, ноутбук их только
импортирует и вызывает.

Модели делятся на две группы по префиксу имени: `hw1_` — модели первой
домашней работы, `hw2_` — новые. В примере старые модели — заглушка
и TF-IDF с логистической регрессией, новые — случайный лес и гистограммный
бустинг на тех же признаках без подбора параметров. Это не образец хорошего
решения, а образец устройства: как назвать модели и что сохранить.

Выводы ниже получены на скрытом тесте. У себя вы запускаете ноутбук
на обучающем файле. Пути заданы в первой ячейке кода, при проверке
мы подставляем свои через переменные окружения.

In [4]:
import os

import pandas as pd

from solution_utils import compute_metrics, init_reaction_models, predict_reaction, read_dialogs, read_labels, save_predictions

In [9]:
TRAIN_PATH = os.environ.get("HW2_TRAIN", "../hw1/hw1_train.jsonl")
DATA_PATH = os.environ.get("HW2_DATA", TRAIN_PATH)
LABELS_PATH = os.environ.get("HW2_LABELS", DATA_PATH)
PREDICTIONS_PATH = os.environ.get("HW2_PREDICTIONS", "predictions.csv")

dialogs = read_dialogs(DATA_PATH)
reaction_labels = read_labels(LABELS_PATH, "reaction") if LABELS_PATH else {}

print("Диалогов:", len(dialogs))

Диалогов: 10000


## Реакция пользователя: старые и новые модели

In [10]:
reaction_models = init_reaction_models(TRAIN_PATH)

predictions, metrics = {}, {}
for name, model in reaction_models.items():
    predictions[name] = predict_reaction(model, dialogs)
    if reaction_labels:
        metrics[name] = compute_metrics(dialogs, predictions[name], reaction_labels)

pd.DataFrame(metrics).T.round(3)

,macro-F1,accuracy
hw1_log_reg,0.790,0.815
hw1_logreg_balanced,0.550,0.570
hw1_mymodel,0.823,0.841
hw1_svc,0.859,0.883
hw1_dmf,0.248,0.591
hw1_ds,0.332,0.449
hw1_nb,0.838,0.872
hw2_bagging,0.771,0.830
hw2_random_forest,0.864,0.886
hw2_lightgbm,0.831,0.854


## Ответы всех моделей

Последняя ячейка сохраняет ответы всех моделей, старых и новых. По этому
файлу мы считаем метрики на скрытом тесте и разность macro-F1 лучшей модели
`hw2_` и лучшей модели `hw1_`.

In [11]:
save_predictions(PREDICTIONS_PATH, dialogs, predictions)
print("Строк в файле ответов:", len(dialogs) * len(predictions))

Строк в файле ответов: 130000
